# מעבדה 02 — משוואות מצב

במעבדה זו תבנו את משוואת המצב של ואן דר ואלס משני תיקונים פיזיקליים לחוק הגזים האידיאליים —
משיכה מולקולרית ונפח מודר — תחקרו את משטח ה-P-v-T שאותם תיקונים מכופפים, ותבדקו את הסיפור
כולו מול נתוני CO2 שפורסמו על ידי NIST Chemistry WebBook.

עברו על המעבדה לפי הסדר. במקומות שבהם המחברת מבקשת מכם לנבא, רשמו את הניבוי שלכם בתא המיועד
לכך **לפני** שתריצו את התא הבא. זה אינו טקס: ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה
לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | כמות קבועה $N$ של חומר דחיס פשוט, המתואר על ידי $(P, v, T)$ כאשר $v = V/N$; הגז האידיאלי הוא המקרה הפרטי $a=b=0$ |
| **דינמיקה** | אין — משוואת מצב סטטית, לא מסלול לאינטגרציה |
| **גבול** | סגור; $N$ קבוע, $v$ ו-$T$ נקבעים מבחוץ, $P$ נקרא ממשוואת המצב |
| **צבר** | לא רלוונטי — תרמודינמיקה מאקרוסקופית |
| **מוזנח** | כל הפירוט המיקרוסקופי; $a$ ו-$b$ הם קבועים פנומנולוגיים |
| **תקף כאשר** | המשטרים הדליל והקלאסי (אידיאלי) ובעל הצפיפות הבינונית סמוך לקריטי (ואן דר ואלס) |
| **אופני כישלון** | אזור הפיתול התת-קריטי $(\partial P/\partial v)_T > 0$ (מודול 14); משטרים מנוונים קוונטית (מודול 17) |

כל הפיזיקה שוכנת ב-`thermolab.gases` — פתחו וקראו. שום דבר בקורס הזה אינו מוסתר בתוך מסגרת
עבודה.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import gases
from thermolab.constants import K_B, N_A
from thermolab.validation import relative_error

# CO2's van der Waals constants, fit from its measured critical point (NIST Chemistry
# WebBook: T_c = 304.13 K, P_c = 7.3773e6 Pa) via gases.vdw_constants_from_critical -- the
# per-particle convention is explained in the module page's "A note on units" box.
CO2_T_C = 304.13  # K
CO2_P_C = 7.3773e6  # Pa
CO2_A, CO2_B = gases.vdw_constants_from_critical(CO2_T_C, CO2_P_C)
v_c, t_c, p_c = gases.vdw_critical_point(CO2_A, CO2_B)

print(f"k_B = {K_B:.6e} J/K")
print(f"CO2 a = {CO2_A:.4e} Pa m^6  (fit from its critical point)")
print(f"CO2 b = {CO2_B:.4e} m^3")
print(f"predicted (v_c, T_c, P_c) = ({v_c:.4e} m^3, {t_c:.2f} K, {p_c:.4e} Pa)")

## לנבא לפני שמחשבים

התחייבו לתשובה לכל אחת מהשאלות הבאות *לפני* שתריצו משהו.

1. אתם מחברים שני בקבוקי גז זהים לאחד — כפליים גז, בכפליים נפח, אותה $T$. אילו מבין $P$,
   $V$, $T$, $N$, $U$ משתנים, ואילו נשארים כשהיו?
2. CO2 בטמפרטורה 280 קלווין נדחס לאט בטמפרטורה קבועה. האם הלחץ ממשיך לטפס, מתייצב, או עושה
   משהו אחר?
3. האם קיימת טמפרטורה שמעליה שום לחץ אינו יכול לנזל גז?
4. שני גזים שונים באותה טמפרטורה *מצומצמת* ובאותו לחץ מצומצם — אותו מקדם דחיסות $Z$, או
   ש-$Z$ תלוי בזהות הגז?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 1 — המשטח האידיאלי, והדלקת גז ממשי

בנו את משטח ה-P-v-T של הגז האידיאלי בעזרת `gases.pvt_surface`, ואז השתמשו במחוונים כדי
להדליק חלק מן ה-$a$ וה-$b$ של CO2 ולצפות בקיפול מופיע. המחוונים משרטטים מחדש בעת השחרור
(`interact_manual`) ולא תוך כדי גרירה — השרטוט מחדש של המשטח, לא הפיזיקה, הוא החלק האיטי.

In [ ]:
import ipywidgets as widgets

v_grid = np.linspace(1.2 * CO2_B, 6.0 * v_c, 35)
t_grid = np.linspace(0.6 * t_c, 1.6 * t_c, 35)
ideal_v, ideal_t, ideal_p = gases.pvt_surface(v_grid, t_grid, 0.0, 0.0)


def show_surface(a_fraction=0.0, b_fraction=0.0):
    a, b = a_fraction * CO2_A, b_fraction * CO2_B
    v_mesh, t_mesh, p_mesh = gases.pvt_surface(v_grid, t_grid, a, b)

    fig = plt.figure(figsize=(7.0, 5.5))
    ax = fig.add_subplot(projection="3d")
    ax.plot_wireframe(
        ideal_v * 1e27, ideal_t, ideal_p / 1e3, color="0.7", linewidth=0.4, rstride=2, cstride=2
    )
    ax.plot_surface(
        v_mesh * 1e27, t_mesh, p_mesh / 1e3, cmap="viridis", alpha=0.9,
        rstride=1, cstride=1, linewidth=0,
    )
    ax.set_xlabel("v (1e-27 m^3)")
    ax.set_ylabel("T (K)")
    ax.set_zlabel("P (kPa)")
    ax.set_title(f"a = {a_fraction:.1f} x CO2's a, b = {b_fraction:.1f} x CO2's b "
                 "(pale wireframe: the ideal surface, a=b=0)")
    plt.show()


widgets.interact_manual(
    show_surface,
    a_fraction=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.0, description="a fraction"),
    b_fraction=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.0, description="b fraction"),
);

## חלק 2 — ניסוי ההכפלה

חַברו שתי דגימות זהות — $N \to 2N$, $V \to 2V$, אותה $T$ — וטבלו את יחס הלפני/אחרי של כל
גודל. זהו הניסוי המפריך עבור התפיסה המוטעית `doubling-doubles-everything`: בדקו אילו יחסים
יוצאים $2$ ואילו יוצאים $1$, מול הניבוי שלכם מחלק 0.

In [ ]:
n_particles, volume, temperature = 5.0e22, 2.0e-3, 300.0

v = volume / n_particles
pressure = gases.van_der_waals_pressure(v, temperature, CO2_A, CO2_B)
internal_energy = 1.5 * n_particles * K_B * temperature  # 3 dof, ideal-gas-like estimate

n_joined, v_joined = 2 * n_particles, 2 * volume
v_per_particle_joined = v_joined / n_joined
pressure_joined = gases.van_der_waals_pressure(v_per_particle_joined, temperature, CO2_A, CO2_B)
internal_energy_joined = 1.5 * n_joined * K_B * temperature

rows = [
    ("N", n_particles, n_joined),
    ("V", volume, v_joined),
    ("T", temperature, temperature),
    ("P", pressure, pressure_joined),
    ("U", internal_energy, internal_energy_joined),
]
print(f"{'quantity':<10}{'base':>16}{'joined':>16}{'ratio':>10}")
for name, base, joined in rows:
    print(f"{name:<10}{base:>16.6e}{joined:>16.6e}{joined / base:>10.4f}")

## חלק 3 — משפחת האיזותרמות סביב $T_c$

`isotherm_family` מציב כמה טמפרטורות על רשת $v$ אחת בבת אחת. סרקו דרך ה-$T_c$ של CO2 ואתרו
את הפיתול מופיע מתחתיה.

In [ ]:
v_grid_3 = np.linspace(1.05 * CO2_B, 4.0 * v_c, 400)
temperatures_3 = np.array([0.85, 0.95, 1.0, 1.05, 1.15, 1.3]) * t_c

pressures_3 = gases.isotherm_family(v_grid_3, temperatures_3, CO2_A, CO2_B)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for temperature, pressure in zip(temperatures_3, pressures_3, strict=True):
    ax.plot(v_grid_3 * 1e27, pressure / 1e6, label=f"T = {temperature:.0f} K")
ax.axvline(v_c * 1e27, color="crimson", ls="--", lw=1.0, label="v_c")
ax.set_xlabel("v (1e-27 m^3)")
ax.set_ylabel("P (MPa)")
ax.set_title("CO2 isotherm family across T_c")
ax.legend()
plt.tight_layout()
plt.show()

## חלק 4 — מדידת $T_c$ בסריקה אחר שטיחות

`vdw_critical_point` נותן את $T_c$ משלוש שורות אלגברה. כאן נ*מדוד* אותו במקום זאת, בדרך
שנסיין חסר צורה סגורה היה נוקט: לסרוק איזותרמות ולמצוא היכן הנקודה השטוחה ביותר על העקומה
(ה-$|\partial P/\partial v|$ הקטן ביותר בכל מקום עליה) מתכווצת בעצמה אל (סביבת) האפס.

מתחת ל-$T_c$ האיזותרמה מפתחת לולאה של ממש, שבה $\partial P/\partial v$ חוצה את האפס בשתי
נקודות *רגילות* — אפס מדויק, לא אות שטיחות, ולא משהו שחיפוש פשוט של "השיפוע הקטן ביותר על
הרשת" יכול להבחין בינו לבין הנקודה הקריטית האמיתית. לכן הסריקה נותנת אמון רק בענף
העל-קריטי, שם האיזותרמה נשארת מונוטונית והנקודה השטוחה ביותר שלה מתכווצת ברציפות לעבר האפס
ככל ש-$T \to T_c^+$, ועוצרת בטמפרטורה הראשונה (הגבוהה ביותר) המשתטחת מתחת לסף קטן.

In [ ]:
def isotherm_min_abs_slope(temperature, v_grid, a, b, h_fraction=1e-4):
    """The smallest |dP/dv| found anywhere along one isotherm, by central differences."""
    h = h_fraction * v_grid
    p_plus = gases.van_der_waals_pressure(v_grid + h, temperature, a, b)
    p_minus = gases.van_der_waals_pressure(v_grid - h, temperature, a, b)
    return float(np.min(np.abs((p_plus - p_minus) / (2.0 * h))))


def scan_for_t_c(n_temperatures=200, threshold_fraction=0.01, n_v=2000):
    """Measure T_c by sweeping down the supercritical branch to the first flat isotherm.

    Returns (measured_t_c, half_grid_spacing). The second value is the DISCRETISATION
    uncertainty only -- Part 8 examines what it leaves out.
    """
    temperatures = np.linspace(1.3 * t_c, 0.7 * t_c, n_temperatures)  # descending
    v_grid = np.linspace(1.2 * CO2_B, 6.0 * v_c, n_v)
    threshold = threshold_fraction * (p_c / v_c)  # p_c/v_c is the characteristic slope

    min_slopes = np.array([isotherm_min_abs_slope(T, v_grid, CO2_A, CO2_B) for T in temperatures])
    first_flat = int(np.argmax(min_slopes < threshold))
    return temperatures[first_flat], 0.5 * abs(temperatures[1] - temperatures[0])


# 200 temperatures x a 2000-point v-grid = 4e5 evaluations, well under a second.
measured_t_c, t_c_uncertainty = scan_for_t_c()

print(f"flatness-scan T_c = {measured_t_c:.2f} +/- {t_c_uncertainty:.2f} K")
print(f"closed-form   T_c = {t_c:.2f} K   (gases.vdw_critical_point)")
print(f"NIST          T_c = {CO2_T_C:.2f} K")
print(f"scan vs NIST relative miss: {relative_error(measured_t_c, CO2_T_C):.3%}")

## חלק 5 — איזותרמת ה-CO2 של NIST: חישוב $Z$

`data/co2-isotherm-280k.csv` מכיל ערכים מייצגים שפורסמו עבור התנהגות הלחץ-נפח האיזותרמית של
CO2 ב-280 קלווין (ראו את סעיף האימות בדף המודול, ואת הערת הכותרת של קובץ ה-CSV עצמו, לגבי
אופן בנייתו). טענו אותו, חשבו $Z = Pv/(k_BT)$ לאורך האיזותרמה, והניחו מעליו את הניבוי
האידיאלי ($Z=1$) ואת ניבוי ואן דר ואלס — המפריך של `ideal-gas-universal`.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "co2-isotherm-280k.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "co2-isotherm-280k.csv")

nist_data = np.loadtxt(csv_path, delimiter=",", comments="#")
nist_temperature, nist_pressure, nist_vm = nist_data[:, 0], nist_data[:, 1], nist_data[:, 2]
nist_v = nist_vm / N_A  # NIST reports molar volume; the course convention is per-particle

z_measured = gases.compressibility_factor(nist_pressure, nist_v, nist_temperature)
ideal_pressure = K_B * nist_temperature / nist_v
vdw_pressure = gases.van_der_waals_pressure(nist_v, nist_temperature, CO2_A, CO2_B)
z_vdw = gases.compressibility_factor(vdw_pressure, nist_v, nist_temperature)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.5))
axes[0].plot(nist_vm * 1e3, nist_pressure / 1e6, "o-", color="#2563eb", label="NIST (measured)")
axes[0].plot(nist_vm * 1e3, ideal_pressure / 1e6, "--", color="0.5", label="ideal")
axes[0].plot(nist_vm * 1e3, vdw_pressure / 1e6, ":", color="#f97316", label="van der Waals")
axes[0].set_xscale("log")
axes[0].set_xlabel("molar volume (1e-3 m^3/mol)")
axes[0].set_ylabel("P (MPa)")
axes[0].set_title("CO2 at 280 K: measured vs ideal vs van der Waals")
axes[0].legend()

axes[1].plot(nist_vm * 1e3, z_measured, "o-", color="#2563eb", label="Z, measured")
axes[1].plot(nist_vm * 1e3, z_vdw, ":", color="#f97316", label="Z, van der Waals")
axes[1].axhline(1.0, ls="--", color="0.5", label="Z = 1 (ideal)")
axes[1].set_xscale("log")
axes[1].set_xlabel("molar volume (1e-3 m^3/mol)")
axes[1].set_ylabel("Z = P v / (k_B T)")
axes[1].set_title("Compressibility factor along the 280 K isotherm")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Z ranges from {z_measured.max():.3f} (dilute) to {z_measured.min():.3f} (compressed "
      "liquid) -- nowhere near the ideal-gas value of 1 once compression is well under way.")

## חלק 6 — מצבים מתאימים: CO2, N2 וארגון מתלכדים

ל-CO2, ל-N2 ולארגון יש $a$ ו-$b$ שונים לחלוטין — ונקודות קריטיות שונות לחלוטין. בעזרת
הקבועים הקריטיים שפורסמו עבורם בלבד (אין צורך בנתוני איזותרמה עבור N2 או ארגון), שרטטו את
האיזותרמה של כל גז ב*אותה* טמפרטורה מצומצמת $T_r$ וצפו בהן מתלכדות לעקומה אחת במשתנים
מצומצמים, בדיוק כפי ש-`vdw_pressure_reduced` מנבא ללא שום קלט האופייני לחומר מסוים.

In [ ]:
species_critical_points = {
    "CO2": (304.13, 7.3773e6),
    "N2": (126.19, 3.3958e6),
    "Ar": (150.9, 4.87e6),
}

v_r_grid = np.linspace(0.5, 4.0, 300)
t_r_shared = 1.1  # the same reduced temperature for every gas

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, (t_c_i, p_c_i) in species_critical_points.items():
    a_i, b_i = gases.vdw_constants_from_critical(t_c_i, p_c_i)
    v_c_i, _t_c_i, _p_c_i = gases.vdw_critical_point(a_i, b_i)
    v_i = v_r_grid * v_c_i
    temperature_i = t_r_shared * t_c_i
    pressure_i = gases.van_der_waals_pressure(v_i, temperature_i, a_i, b_i)
    p_r_i, v_r_i, _t_r_i = gases.reduced_variables(pressure_i, v_i, temperature_i, a_i, b_i)
    ax.plot(v_r_i, p_r_i, lw=2.5, alpha=0.8, label=name)

p_r_universal = gases.vdw_pressure_reduced(v_r_grid, t_r_shared)
ax.plot(v_r_grid, p_r_universal, "k--", lw=1.2, label="vdw_pressure_reduced (parameter-free)")
ax.set_xlabel("v_r = v / v_c")
ax.set_ylabel("P_r = P / P_c")
ax.set_title(f"Corresponding states at T_r = {t_r_shared}: three gases, one curve")
ax.legend()
plt.tight_layout()
plt.show()

## חלק 7 — בדיקות אוטומטיות

מקצת מן הטענות שעליהן נשען הסיפור של המחברת הזו, משוקפות מערכת המבחנים של הפרויקט.

In [ ]:
from thermolab import kinetics, paths

# 1. The ideal-gas limit: a = b = 0 matches gases.ideal_gas_pressure exactly. The reference
#    comes from the library, not from a formula retyped here -- v = V/N = 1e-3/1000 = 1e-6.
ideal_ref = gases.ideal_gas_pressure(1000, 300.0, 1e-3)
vdw_zero = gases.van_der_waals_pressure(1e-6, 300.0, 0.0, 0.0)
assert relative_error(float(vdw_zero), ideal_ref) < 1e-12

# 2. The closed-form critical point matches the pressure formula evaluated there.
p_at_critical = gases.van_der_waals_pressure(v_c, t_c, CO2_A, CO2_B)
assert relative_error(float(p_at_critical), p_c) < 1e-10

# 3. Z_c = 3/8, exactly, for every van der Waals substance.
z_c = gases.compressibility_factor(p_c, v_c, t_c)
assert relative_error(float(z_c), 3.0 / 8.0) < 1e-12

# 4. The C4a dedup: kinetics.py and paths.py re-import gases.py's functions by identity.
assert kinetics.ideal_gas_pressure is gases.ideal_gas_pressure
assert paths.ideal_gas_pressure is gases.ideal_gas_pressure

print("all checks passed")

## חלק 8 — מדידה: ציטוט $T_c$ עם האי-ודאות שלו

מספר ללא אי-ודאות אינו מדידה. סריקת השטיחות מחלק 4 נותנת אחת; התא הזה מנסח את התוצאה כפי
שדוח ניסויי היה עושה — ואז שואל את השאלה הקשה יותר, והיא האם תחום השגיאה מודד את כל מה
שעלול להשתבש.

In [ ]:
agrees = abs(measured_t_c - CO2_T_C) <= 3.0 * t_c_uncertainty

print(f"T_c(CO2) = {measured_t_c:.2f} +/- {t_c_uncertainty:.2f} K   (flatness scan, this notebook)")
print(f"T_c(CO2) = {CO2_T_C:.2f} K                     (NIST Chemistry WebBook, reference)")
print(f"T_c(CO2) = {t_c:.2f} K                     (gases.vdw_critical_point, closed form)")
print()
print(f"The scan's own uncertainty {'comfortably covers' if agrees else 'does not cover'} "
      "the gap to the NIST reference.")
print()
print("But read that error bar carefully: +/- half a grid spacing covers the DISCRETISATION "
      "only. The flatness threshold adds a separate BIAS -- the scan stops when the slope "
      "falls below the threshold, not when it truly vanishes -- and that bias does not shrink "
      "with the temperature grid. Refine the grid alone and the quoted uncertainty falls while "
      "the bias stays put, so the same measurement drifts from 1 sigma to many sigma away from "
      "the closed form: an error bar that has quietly stopped being honest. Try it below.")

In [ ]:
# Refine the temperature grid ALONE, holding the flatness threshold fixed, and watch the
# quoted uncertainty shrink while the error does not.
print("threshold held at 1% of p_c/v_c:")
print(f"{'n_T':>6}{'T_c meas':>12}{'+/-':>8}{'error':>9}{'sigma':>8}")
for n_temperatures in (200, 400, 800, 1600):
    measured, uncertainty = scan_for_t_c(n_temperatures=n_temperatures)
    error = measured - t_c
    print(f"{n_temperatures:>6}{measured:>12.3f}{uncertainty:>8.3f}{error:>+9.3f}"
          f"{abs(error) / uncertainty:>8.1f}")

# Now refine BOTH together -- the threshold scaled down in step with the grid.
print("\nthreshold tightened in step with the grid:")
print(f"{'n_T':>6}{'threshold':>12}{'T_c meas':>12}{'+/-':>8}{'error':>9}{'sigma':>8}")
for n_temperatures, threshold_fraction in ((200, 0.01), (400, 0.005), (800, 0.0025),
                                           (1600, 0.00125)):
    measured, uncertainty = scan_for_t_c(n_temperatures, threshold_fraction)
    error = measured - t_c
    print(f"{n_temperatures:>6}{threshold_fraction:>12.5f}{measured:>12.3f}{uncertainty:>8.3f}"
          f"{error:>+9.3f}{abs(error) / uncertainty:>8.1f}")

print("\nThe first table is the cautionary one: the error bar shrinks, the error does not, and "
      "a measurement that started out honest ends up confidently wrong. The second converges "
      "properly -- error and uncertainty falling together, the scan staying within about one "
      "sigma of the closed form the whole way down.")

## לבדוק את ההבנה

הריצו את התא שלהלן לשאלון עם בדיקה אוטומטית. אותן שאלות עצמן, עם הסברים כתובים לכל אפשרות,
נמצאות בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "02-equations-of-state.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. סטודנט אומר: "גזים ממשיים סוטים מחוק הגזים האידיאליים רק משום שהמולקולות שלהם תופסות
   מקום." שפרו את המשפט הזה כך שיהיה נכון באמת, ואמרו במדויק מה הוא משמיט.
2. הסבירו, בלי משוואות, מדוע הכפלה של כמות גז ממשי ושל נפח המיכל שלו גם יחד, באותה
   טמפרטורה, מותירה את הלחץ שלו ללא שינוי.
3. מה מדידת סריקת השטיחות של $T_c$ שביצעתם היום *אינה* מבססת לגבי מודל ואן דר ואלס בסביבת
   הנקודה הקריטית?

**התשובות שלכם:**

1.
2.
3.